In [132]:
!pip install seaborn matplotlib scanpy numpy anndata pandas adjustText scipy

In [133]:
# -- Load libraries
import os
import scanpy as sc
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from scipy.stats import kruskal, mannwhitneyu
import itertools
from adjustText import adjust_text

In [134]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [135]:
# -- Color Palettes
patient_colors_list = [
    sns.husl_palette(n_colors=1, h=h, s=0.9, l=0.65).as_hex()[0]
    for h in np.linspace(0.01, 0.70, 14)
]

# map to patient IDs
patient_ids = ["C01", "C02", "C03",
               "E01", "E02", "E03", "E04", "E05",
               "E06", "E07", "E08", "E09", "E10", "E11"]

patient_palette = dict(zip(patient_ids, patient_colors_list))

tissue_palette = {
    "Ctrl": "#7FA25C",
    "EuE":  "#028090",
    "EcP":  "#456990",
    "EcO":  "#CE7DA5"
}

immune_palette = {
    # -- Macrophages / Monocytes (warm tones)
    "Classical Monocytes": "#E07B54",
    "Non-Classical Monocytes":"#F2A65A",
    "Tissue Resident Macrophages": "#C14F3A",

    # -- Dendritic cells (purples)
    "cDC1": "#7B4FA6",
    "cDC2": "#A87DC2",
    "pDC": "#C9A8E0",

    # -- NK cells (greens)
    "CD16+ NK cells": "#2D8C6E",
    "CD16- NK cells": "#6DBF9E",

    # -- T cells (blues)

    "CD4 T cells": "#0B4A7E",
    "CD8 T cells": "#2E6FA3",
    "Tregs": "#5B9EC9",
    "γδ T cells": "#A8CBE0",


    # -- B cells (yellow)
    "B cells": "#E8C84A"
}

# -- Setting Tissue Type Hue Order
hue_order_tissue = ["Ctrl", "EuE", "EcO", "EcP"]

In [136]:
# -- Paths
input_path = "/content/drive/MyDrive/endo-immune-atlas/data/processed/GSE179640_immune_subset.h5ad"
output_path_data = "/content/drive/MyDrive/endo-immune-atlas/data/processed"
output_path_figures = "/content/drive/MyDrive/endo-immune-atlas/results/final_figures"

os.makedirs(output_path_figures, exist_ok=True)

In [137]:
# -- Import object
immune_obj = sc.read_h5ad(input_path)

In [138]:
# -- Filter out Proliferating group
immune_obj = immune_obj[
    immune_obj.obs["immune_cell_type"] != "Proliferating"
].copy()

In [139]:
#==========================================================================
# SENESCENT MARKERS
#==========================================================================
# - Core/Checkpoint Inbibitors
core_sen_markers = ["CDKN1A",
                    "CDKN2A",
                    "CDKN2B",
                    "TP53",
                    "RB1",
                    "HMGA1"]

# - SASP
sasp_markers = ["IL6",
                "CXCL8",
                "IL1B",
                "TNF",
                "MMP3",
                "MMP9",
                "CXCL1",
                "CXCL2",
                "CXCL10",
                "CCL2",
                "CCL5",
                "VEGFA",
                "IGFBP3",
                "IGFBP7"]

# - Innate-Specific
innate_sen_markers = ["MARCO",
                      "APOE",
                      "SPP1",
                      "CXCR2",
                      "GLB1",
                      "PTPRC",
                      "CD68",
                      "TIGIT",
                      "CD44"]

# - Adaptive-Specific
adaptive_sen_markers = ["B3GAT1",
                        "KLRG1",
                        "EOMES",
                        "TBX21",
                        "LAG3",
                        "PDCD1"]


#==========================================================================
# EXHAUSTION/DYSFUNCTION MARKERS
#==========================================================================
t_cell_exhaustion_markers = ["PDCD1",
                             "HAVCR2",
                             "LAG3",
                             "TIGIT",
                             "TOX",
                             "TOX2",
                             "NR4A1",
                             "CXCL13",
                             "ENTPD1",
                             "KLRG1",
                             "ST2"]

nk_exhaustion_markers = ["TIGIT",
                         "LAG3",
                         "PDCD1",
                         "KIR2DL1",
                         "KIR2DL2",
                         "KIR3DL1",
                         "HAVCR2",
                         "CD96",
                         "KLRC1"]

mac_dysfunction_markers = ["MARCO",
                           "CD274",
                           "SIGLEC1",
                           "MERTK",
                           "IL10",
                           "TGFB1",
                           "VSIG4"]

b_cell_exhaustion_markers = ["PDCD1",
                             "TOX",
                             "FAS",
                             "CD39",
                             "FCRL4",
                             "FCRL5"]

dc_dysfunction = {
    "cDC1": ["IDO1", "CD274", "LAG3", "TIGIT"],
    "cDC2": ["CD274", "IL10", "TIGIT", "FCGR2B"],
    "pDC":  ["CD274", "TIGIT", "LAG3", "LILRB4"]
}

In [140]:
#==========================================================================
# SUBSET IMMUNE CELL COMPARTMENTS/TYPES FOR SCORING
#==========================================================================

# -- Immune Cell Compartments
innate_cells = [
    "Classical Monocytes",
    "Non-Classical Monocytes",
    "Tissue Resident Macrophages",
    "cDC1",
    "cDC2",
    "pDC",
    "CD16+ NK cells",
    "CD16- NK cells",
]

adaptive_cells = [
    "CD4 T cells",
    "CD8 T cells",
    "Tregs",
    "γδ T cells",
    "B cells"
]

# -- Cell types

cell_type_map = {
    "t_cells": ["CD4 T cells", "CD8 T cells", "Tregs", "γδ T cells"],
    "nk_cells": ["CD16+ NK cells", "CD16- NK cells"],
    "myeloids": ["Classical Monocytes", "Non-Classical Monocytes", "Tissue Resident Macrophages"],
    "b_cells": ["B cells"],
    "DCs": ["cDC1", "cDC2", "pDC"]

}

In [141]:
#==========================================================================
# CELL COUNTS PER CELL TYPE
#==========================================================================
cell_counts = (
    immune_obj.obs
    .groupby(["tissue_type", "immune_cell_type"], observed=False)
    .size()
    .reset_index(name="n_cells")
)

cell_counts_long = cell_counts.pivot(index="immune_cell_type",
                                     columns="tissue_type",
                                     values="n_cells"
).fillna(0)

cell_counts_long = cell_counts_long.reindex(columns=["Ctrl", "EuE", "EcO", "EcP"])

ax = plt.figure(figsize=(8, 6))
sns.heatmap(
    cell_counts_long,
    annot=True,
    fmt="g",
    cmap="YlGnBu",
    linewidths=0.5
)

plt.title("Cell Counts by Tissue Type")
plt.xlabel("")
plt.ylabel("")
plt.tight_layout()

plt.savefig(
    os.path.join(output_path_figures, "06_cell_counts_per_tissue_cell_type.png"),
    bbox_inches='tight',
    dpi=300
)
plt.close()

In [142]:
# -- Add lineage column for tissue type grouping per immune cell lineage
immune_obj.obs["lineage"] = immune_obj.obs["immune_cell_type"].apply(
    lambda x: "Innate" if x in innate_cells else "Adaptive"
)

In [143]:
#==========================================================================
# MAKE OBJECTS FOR IMMUNE CELL TYPES - FOR DYSFUNCTION SCORING
#==========================================================================
def subset_immune(immune_obj, cell_types):
    return immune_obj[immune_obj.obs["immune_cell_type"].isin(cell_types)].copy()

immune_cell_subsets = {name: subset_immune(immune_obj, cells) for name, cells in cell_type_map.items()}

In [144]:
#==========================================================================
# SENESCENCE SCORING
#==========================================================================

def scoring(adata, score_markers, score_name):
  return sc.tl.score_genes(adata, score_markers, score_name = score_name)

scoring(immune_obj, core_sen_markers, "core_sen_score")
scoring(immune_obj, sasp_markers, "sasp_score")

innate_obj = subset_immune(immune_obj, innate_cells)
scoring(innate_obj, innate_sen_markers, "innate_sen_score")
immune_obj.obs.loc[innate_obj.obs.index, "innate_sen_score"] = innate_obj.obs["innate_sen_score"]

adaptive_obj = subset_immune(immune_obj, adaptive_cells)
scoring(adaptive_obj, adaptive_sen_markers, "adaptive_sen_score")
immune_obj.obs.loc[adaptive_obj.obs.index, "adaptive_sen_score"] = adaptive_obj.obs["adaptive_sen_score"]

In [145]:
# ==========================================================================
# FIGURE 6_1: SEN Score on UMAP
# ==========================================================================
centroids = pd.DataFrame(
    immune_obj.obsm["X_umap"],
    index=immune_obj.obs_names,
    columns=["UMAP1", "UMAP2"]
)
centroids["immune_cell_type"] = immune_obj.obs["immune_cell_type"].values
centroids = centroids.groupby("immune_cell_type", observed = False)[["UMAP1", "UMAP2"]].mean()

fig = sc.pl.umap(
    immune_obj,
    color=["core_sen_score", "sasp_score", "innate_sen_score", "adaptive_sen_score"],
    ncols=2,
    cmap="RdYlBu_r",
    title=["Core Senescence Score", "SASP Score", "Innate Senescence Score", "Adaptive Senescence Score"],
    show=False,
    return_fig=True
)

from adjustText import adjust_text

umap_axes = [ax for ax in fig.axes if ax.get_xlabel() == "UMAP1"]

for ax in umap_axes:
    texts = []
    for cell_type, row in centroids.iterrows():
        texts.append(
            ax.text(row["UMAP1"], row["UMAP2"], cell_type,
                    fontsize=5, ha="center", va="center", fontweight="bold")
        )
    adjust_text(texts, ax=ax)

plt.savefig(
    os.path.join(output_path_figures, "06_umap_sen_scores.png"),
    bbox_inches='tight',
    dpi=300
)
plt.close()

In [146]:
# ==========================================================================
# FIGURE 6_2: Heat Maps of SEN Score per Tissue Typer per SEN type
# ==========================================================================
score_cell_map = {
    "core_sen_score":     (innate_cells + adaptive_cells, "Core Senescence"),
    "sasp_score":         (innate_cells + adaptive_cells, "SASP"),
    "innate_sen_score":   (innate_cells, "Innate Senescence"),
    "adaptive_sen_score": (adaptive_cells, "Adaptive Senescence")
}

cell_type_summary = immune_obj.obs.groupby(
["immune_cell_type", "tissue_type"],
                                  observed=False
)[["core_sen_score", "sasp_score",
   "innate_sen_score", "adaptive_sen_score"]].mean().reset_index()

for score_col, (cell_order, title) in score_cell_map.items():
    pivot = cell_type_summary.pivot(
        index="immune_cell_type",
        columns="tissue_type",
        values=score_col
    ).reindex(cell_order)

    fig, ax = plt.subplots(figsize=(6, len(cell_order) * 0.6))
    sns.heatmap(
        pivot,
        cmap="RdYlBu_r",
        center=0,
        ax=ax,
        linewidths=0.5,
        cbar_kws={"label": "Mean Score"}
    )
    ax.set_title(title)
    ax.set_xlabel("Tissue Type")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.savefig(
        os.path.join(output_path_figures, f"06_heatmap_{score_col}.png"),dpi=300, bbox_inches="tight"
    )
    plt.close()

In [147]:
# ==========================================================================
# FIGURE 6_3: Violin plot - Types of SEN Score per Cell Type -- edit bottom graph
# ==========================================================================

score_cell_map = {
    "Core Senescence":   ("core_sen_score",     innate_cells + adaptive_cells),
    "SASP":              ("sasp_score",          innate_cells + adaptive_cells),
    "Innate Senescence": ("innate_sen_score",    innate_cells),
    "Adaptive Senescence": ("adaptive_sen_score", adaptive_cells)
}
fig, axes = plt.subplots(2, 2, figsize=(30, 20))

for ax, (title, (score, cell_order)) in zip(axes.flatten(), score_cell_map.items()):
    subset = immune_obj.obs[immune_obj.obs["immune_cell_type"].isin(cell_order)]
    sns.violinplot(
        data=subset,
        x="immune_cell_type",
        y=score,
        hue="immune_cell_type",
        palette=immune_palette,
        legend=False,
        order=cell_order,
        ax=ax
    )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(f"{title} Score")
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()

plt.savefig(
    os.path.join(output_path_figures, "06_senescence_type_per_cell_type.png"),
    dpi=300, bbox_inches="tight"
    )
plt.close()


In [162]:
# ==========================================================================
# FIGURE 6_4: SEN Score Typer per Tissue Type
# ==========================================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.violinplot(
    data=immune_obj.obs,
    x="tissue_type",
    y="core_sen_score",
    hue="tissue_type",
    legend=False,
    palette=tissue_palette,
    order=["Ctrl", "EuE", "EcP", "EcO"],
    ax=axes[0]
)
axes[0].set_title("Core Senescence Score")
axes[0].set_ylabel("Core Senescence Score")
axes[0].set_xlabel("")


plt.savefig(
    os.path.join(output_path_figures, "06_senescence_type_per_cell_type_core_sasp.png"),
    dpi=300, bbox_inches="tight"
    )
plt.close()


sns.violinplot(
    data=immune_obj.obs,
    x="tissue_type",
    y="sasp_score",
    hue="tissue_type",
    legend=False,
    palette=tissue_palette,
    order=["Ctrl", "EuE", "EcP", "EcO"],
    ax=axes[1]
)
axes[1].set_title("SASP Senescense Score")
axes[1].set_ylabel("SASP Senescence Score")
axes[1].set_xlabel("")



fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.violinplot(
    data=immune_obj.obs[immune_obj.obs["lineage"] == "Innate"],
    x="tissue_type",
    y="innate_sen_score",
    hue="tissue_type",
    legend=False,
    palette=tissue_palette,
    order=["Ctrl", "EuE", "EcP", "EcO"],
    ax=axes[0]
)
axes[0].set_title("Innate Senescence Score\n(Innate Immune Cells)")
axes[0].set_ylabel("Innate Senescence Score")
axes[0].set_xlabel("")

sns.violinplot(
    data=immune_obj.obs[immune_obj.obs["lineage"] == "Adaptive"],
    x="tissue_type",
    y="adaptive_sen_score",
    hue="tissue_type",
    legend=False,
    palette=tissue_palette,
    order=["Ctrl", "EuE", "EcP", "EcO"],
    ax=axes[1]
)
axes[1].set_title("Adaptive Senescence Score\n(Adaptive Immune Cells)")
axes[1].set_ylabel("Adaptive Senescence Score")
axes[1].set_xlabel("")

plt.savefig(
    os.path.join(output_path_figures, "06_senescence_type_per_cell_type_immune.png"),
    dpi=300, bbox_inches="tight"
    )
plt.close()



In [149]:
# -- Create a composite score for each lineage and assign to obj

# innate composite
immune_obj.obs["innate_composite_score"] = (
    immune_obj.obs["core_sen_score"] +
    immune_obj.obs["sasp_score"] +
    immune_obj.obs["innate_sen_score"]
) / 3

# adaptive composite
immune_obj.obs["adaptive_composite_score"] = (
    immune_obj.obs["core_sen_score"] +
    immune_obj.obs["sasp_score"] +
    immune_obj.obs["adaptive_sen_score"]
) / 3

# assign per cell
immune_obj.obs["composite_sen_score"] = immune_obj.obs.apply(
    lambda row: row["innate_composite_score"]
    if row["immune_cell_type"] in innate_cells
    else row["adaptive_composite_score"],
    axis=1
)

In [150]:
#==========================================================================
# EXHAUSTION/DYSFUNCTION SCORING
#==========================================================================
dysfunction_map = {
    "t_cells": {
        "markers": t_cell_exhaustion_markers,
        "score_name": "tcell_exhaustion_score"
    },
    "nk_cells": {
        "markers": nk_exhaustion_markers,
        "score_name": "nk_dysfunction_score"
    },
    "myeloids": {
        "markers": mac_dysfunction_markers,
        "score_name": "macrophage_dysfunction_score"
    },
    "b_cells": {
        "markers": b_cell_exhaustion_markers,
        "score_name": "bcell_exhaustion_score"
    }
}

for subset_name, params in dysfunction_map.items():
    subset = immune_cell_subsets[subset_name]

    scoring(
        subset,
        params["markers"],
        params["score_name"]
    )

    immune_obj.obs.loc[
        subset.obs.index,
        params["score_name"]
    ] = subset.obs[params["score_name"]]


dc_map = {
    "cDC1": {
        "markers": dc_dysfunction["cDC1"],
        "score_name": "cdc1_dysfunction_score"
    },
    "cDC2": {
        "markers": dc_dysfunction["cDC2"],
        "score_name": "cdc2_dysfunction_score"
    },
    "pDC": {
        "markers": dc_dysfunction["pDC"],
        "score_name": "pdc_dysfunction_score"
    }
}

for dc_type, params in dc_map.items():

    dc_subset = subset_immune(immune_obj, [dc_type])

    scoring(
        dc_subset,
        params["markers"],
        params["score_name"]
    )

    immune_obj.obs.loc[
        dc_subset.obs.index,
        params["score_name"]
    ] = dc_subset.obs[params["score_name"]]


immune_obj.obs["dysfunction_score"] = np.nan


score_map = {
    "t_cells": "tcell_exhaustion_score",
    "nk_cells": "nk_dysfunction_score",
    "myeloids": "macrophage_dysfunction_score",
    "b_cells": "bcell_exhaustion_score"
}

immune_obj.obs["dysfunction_score"] = np.nan

for group, score_col in score_map.items():

    mask = immune_obj.obs["immune_cell_type"].isin(
        cell_type_map[group]
    )

    immune_obj.obs.loc[mask, "dysfunction_score"] = (
        immune_obj.obs.loc[mask, score_col]
    )

dc_score_map = {
    "cDC1": "cdc1_dysfunction_score",
    "cDC2": "cdc2_dysfunction_score",
    "pDC": "pdc_dysfunction_score"
}

for dc_type, score_col in dc_score_map.items():

    mask = immune_obj.obs["immune_cell_type"] == dc_type

    immune_obj.obs.loc[mask, "dysfunction_score"] = (
        immune_obj.obs.loc[mask, score_col]
    )

In [151]:
# ==========================================================================
# FIGURE 6_5: Heat Map - Dysfunction score per cell type per tissue type
# ==========================================================================
pivot = immune_obj.obs.groupby(
    ["immune_cell_type", "tissue_type"], observed = False
)["dysfunction_score"].mean().unstack()

pivot = pivot.reindex(innate_cells + adaptive_cells)
pivot = pivot.reindex(columns=["Ctrl", "EuE", "EcP", "EcO"])

fig, ax = plt.subplots(figsize=(6, 10))
sns.heatmap(
    pivot,
    cmap="RdYlBu_r",
    center=0,
    ax=ax,
    linewidths=0.5,
    cbar_kws={"label": "Mean Dysfunction Score"}
)
ax.set_title("Dysfunction Score per Cell Type across Tissue Types")
ax.set_xlabel("Tissue Type")
plt.tight_layout()

plt.savefig(
    os.path.join(output_path_figures, f"06_dysfunction_heatmap.png"),
    dpi=300, bbox_inches="tight"
    )
plt.close()

In [152]:
# ==========================================================================
# FIGURE 6_7: SEN vs. DYSFUNCTION/EXHAUSTION
# ==========================================================================

sen_score_map = {
    ct: "innate_sen_score" if ct in innate_cells else "adaptive_sen_score"
    for ct in innate_cells + adaptive_cells
}


immune_obj.obs["plot_sen_score"] = immune_obj.obs.apply(
    lambda row: row["innate_sen_score"]
    if row["immune_cell_type"] in innate_cells
    else row["adaptive_sen_score"],
    axis=1
)


# -- Spearman correlation per cell type
spearman_sen_dys_results = []
for cell_type, sen_score in sen_score_map.items():
    subset = immune_obj.obs[
        immune_obj.obs["immune_cell_type"] == cell_type
    ].dropna(subset=[sen_score, "dysfunction_score"])

    if len(subset) < 30:
        continue

    corr, pval = spearmanr(subset[sen_score], subset["dysfunction_score"])
    spearman_sen_dys_results.append({
        "cell_type":   cell_type,
        "sen_score":   sen_score,
        "spearman_r":  corr,
        "pval":        pval
    })


# -- Visualize
fig, axes = plt.subplots(2, 1, figsize=(50, 20))

# scatter
sns.scatterplot(
    data=immune_obj.obs.dropna(subset=["dysfunction_score"]),
    x="plot_sen_score",
    y="dysfunction_score",
    hue="immune_cell_type",
    palette=immune_palette,
    s=20,
    alpha=0.4,
    ax=axes[0]
)
axes[0].axvline(x=0, color="grey", linewidth=0.8, linestyle="--")
axes[0].axhline(y=0, color="grey", linewidth=0.8, linestyle="--")
axes[0].set_xlabel("Senescence Score")
axes[0].set_ylabel("Dysfunction Score")
axes[0].set_title("Senescence vs Dysfunction")

# ==========================================================================
# Spearman_r, SEN. vs Dysfunc.
# ==========================================================================

sen_vs_dys_spearman_df = pd.DataFrame(spearman_sen_dys_results).sort_values("spearman_r", ascending=False)


sen_vs_dys_spearman_df["pval_adj"] = multipletests(sen_vs_dys_spearman_df["pval"], method="fdr_bh")[1]
sen_vs_dys_spearman_df["significant"] = sen_vs_dys_spearman_df["pval_adj"] < 0.05



# spearman bar plot
sns.barplot(
    data=sen_vs_dys_spearman_df,
    x="cell_type",
    y="spearman_r",
    hue="cell_type",
    legend=False,
    palette=immune_palette,
    ax=axes[1]
)
axes[1].axhline(y=0, color="black", linewidth=0.8)
axes[1].set_ylabel("Spearman r")
axes[1].set_xlabel("")
axes[1].set_title("Senescence vs Dysfunction Correlation per Cell Type")


plt.tight_layout()


plt.savefig(
    os.path.join(output_path_figures, f"06_sen_dysfunction_spearman_corr.png"),
    dpi=300, bbox_inches="tight"
    )
plt.close()

In [161]:
# ==========================================================================
# SPLIT SCORING INTO HIGH AND LOW
# ==========================================================================
# -- Senescent Scores
immune_obj.obs["senescent_state"] = (
    immune_obj.obs["composite_sen_score"] >
    immune_obj.obs["composite_sen_score"].quantile(0.75)
).map(
    {True: "High", False: "Low"}
)

# -- Dysfunction Scores
immune_obj.obs["dysfunction_state"] = (
    immune_obj.obs["dysfunction_score"] >
    immune_obj.obs["dysfunction_score"].quantile(0.75)
).map({True: "High", False: "Low"})



# ==========================================================================
# SEN & DYSFUNCTION CO-OCCURANCE
# ==========================================================================

# -- SEN-high and Dysfunction-high co-occurrence per cell type per tissue type
# create co-occurrence label

immune_obj.obs["sen_dysfunction_label"] = "Other"
immune_obj.obs.loc[
    (immune_obj.obs["senescent_state"] == "High") &
    (immune_obj.obs["dysfunction_state"] == "High"),
    "sen_dysfunction_label"
] = "SEN-high & Dysfunction-high"

immune_obj.obs.loc[
    (immune_obj.obs["senescent_state"] == "High") &
    (immune_obj.obs["dysfunction_state"] == "Low"),
    "sen_dysfunction_label"
] = "SEN-high only"

immune_obj.obs.loc[
    (immune_obj.obs["senescent_state"] == "Low") &
    (immune_obj.obs["dysfunction_state"] == "High"),
    "sen_dysfunction_label"
] = "Dysfunction-high only"


label_palette = {
    "SEN-high & Dysfunction-high": "#C14F3A",
    "SEN-high only":               "#E07B54",
    "Dysfunction-high only":       "#5B9EC9",
    "Other":                       "#D3D3D3"
}

legend_order = ["Dysfunction-high only", "SEN-high only", "SEN-high & Dysfunction-high", "Other"]

immune_obj.obs["sen_dysfunction_label"] = immune_obj.obs["sen_dysfunction_label"].astype("category")


immune_obj.obs["sen_dysfunction_label"] = immune_obj.obs["sen_dysfunction_label"].cat.reorder_categories(legend_order, ordered=True)


# ==========================================================================
# PROPORTIONAL CALCULATIONS
# ==========================================================================

# proportion of SEN-high cells per tissue type per cell type
proportion_sen_high = immune_obj.obs.groupby(
    ["tissue_type", "immune_cell_type"], observed=False)["senescent_state"].apply(
        lambda x: (x == "High").mean()
        ).reset_index()
proportion_sen_high.columns = ["tissue_type", "immune_cell_type", "prop_sen_high"]

# proportion of SEN-high cells per tissue type per cell type
proportion_dys_high = immune_obj.obs.groupby(
    ["tissue_type", "immune_cell_type"], observed=False)["dysfunction_state"].apply(
        lambda x: (x == "High").mean()
        ).reset_index()
proportion_dys_high.columns = ["tissue_type", "immune_cell_type", "prop_dys_high"]


# proportion of SEN-high and DYS-high (co-occurance) per cell type per tissue type
proportion_co_occurance = immune_obj.obs.groupby(
    ["tissue_type", "immune_cell_type"], observed=False
)["sen_dysfunction_label"].apply(
    lambda x: (x == "SEN-high & Dysfunction-high").mean()
).reset_index()
proportion_co_occurance.columns = ["tissue_type", "immune_cell_type", "prop_co_occurrence"]


prop_summary = (
    proportion_sen_high
    .merge(
        proportion_dys_high,
        on=["immune_cell_type", "tissue_type"]
    )
    .merge(
        proportion_co_occurance,
        on=["immune_cell_type", "tissue_type"]
    )
)


proportion_co_occurance["tissue_type"] = pd.Categorical(
    proportion_co_occurance["tissue_type"],
    categories=hue_order_tissue,
    ordered=True
)

# ==========================================================================
# FIGURE 6_8: PROPORTION OF SEN/DYSFUNCTION/EXHAUSTION CO-OCCURANCE
# ==========================================================================

plt.figure(figsize=(10, 10))

ax =sns.scatterplot(
    data=proportion_co_occurance,
    x="tissue_type",
    y="immune_cell_type",
    size="prop_co_occurrence",
    hue="prop_co_occurrence",
    sizes=(20, 400),
    palette="Reds",
    legend="brief"
)

plt.title("Senescence and Dysfunction Co-occurrence")


ax.legend(
    title="Proportion",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    borderaxespad=0,
)

ax.set_ylabel("")
ax.set_xlabel("")

plt.tight_layout()

plt.savefig(
    os.path.join(output_path_figures, "06_proportion_sen&dysfunctional.png"),
    dpi=300, bbox_inches="tight"
    )
plt.close()

In [154]:
# ==========================================================================
# DEG ANALYSIS - SENESCENCE
# ==========================================================================

# -- DEG between SEN-high and SEN-low per cell type

# -- fix this with updated SEN-high
deg_results_sen = {}

for cell_type in immune_obj.obs["immune_cell_type"].unique():
    subset = immune_obj[immune_obj.obs["immune_cell_type"] == cell_type].copy()

    if subset.obs["senescent_state"].value_counts().min() < 10:
        continue

    sc.tl.rank_genes_groups(
        subset,
        groupby="senescent_state",
        groups=["High"],
        reference="Low",
        method="wilcoxon",
        key_added="deg_sen"
    )

    deg_results_sen[cell_type] = sc.get.rank_genes_groups_df(
        subset,
        group="High",
        key="deg_sen"
    )

# -- check
for cell_type, df in deg_results_sen.items():
    print(f"\n{cell_type}")
    print(df[df["pvals_adj"] < 0.05].head(10)[["names", "logfoldchanges", "pvals_adj"]])


# -- DEG volcano plot SEN HIGH VS. LOW per cell type

for cell_type, df in deg_results_sen.items():

    df["-log10_pval"] = -np.log10(df["pvals_adj"] + 1e-300)
    df["significant"] = (df["pvals_adj"] < 0.05) & (abs(df["logfoldchanges"]) > 1.5)

    fig, ax = plt.subplots(figsize=(8, 6))

    ax.scatter(
        df[~df["significant"]]["logfoldchanges"],
        df[~df["significant"]]["-log10_pval"],
        s=3, alpha=0.4, color="grey"
    )
    ax.scatter(
        df[df["significant"]]["logfoldchanges"],
        df[df["significant"]]["-log10_pval"],
        s=5, alpha=0.7, color="#E07B54"
    )

    # label top genes
    top = df[df["significant"]].nlargest(10, "-log10_pval")
    texts = [ax.text(row["logfoldchanges"], row["-log10_pval"],
                     row["names"], fontsize=7) for _, row in top.iterrows()]
    adjust_text(texts, arrowprops=dict(arrowstyle="-", color="black", lw=0.5))

    ax.axvline(x=0.5,  color="red", linestyle="--", linewidth=0.8)
    ax.axvline(x=-0.5, color="red", linestyle="--", linewidth=0.8)
    ax.axhline(y=-np.log10(0.05), color="blue", linestyle="--", linewidth=0.8)

    ax.set_xlabel("Log2 Fold Change")
    ax.set_ylabel("-log10 adjusted p-value")
    ax.set_title(f"SEN-high vs SEN-low DEGs\n{cell_type}")

    plt.savefig(
    os.path.join(output_path_figures, f"06_{cell_type}_SEN_DEG_volcano.png"),
    dpi=300, bbox_inches="tight"
    )
    plt.close()


Classical Monocytes
      names  logfoldchanges      pvals_adj
0    CDKN1A        9.828748  1.485479e-267
1      JUNB        0.393532   1.576049e-17
2     NAMPT        0.574447   6.919441e-16
3     SOCS3        1.181382   1.026256e-11
4  HLA-DPB1        0.306152   3.950790e-11
5       FOS        0.366341   6.898624e-10
6      IL1B        1.205615   4.798816e-09
7      IER5        0.497341   1.072183e-08
8  HLA-DPA1        0.228305   1.873365e-08
9  HLA-DQA1        0.574706   2.174062e-08

CD8 T cells
   names  logfoldchanges      pvals_adj
0  KLRG1        2.181530  1.316476e-171
1  EOMES        2.286883  4.193336e-129
2   GZMK        0.901507   1.440551e-66
3   CST7        0.402076   1.224474e-38
4   CD74        0.451224   3.266084e-27
5  PDCD1        1.556672   2.130154e-26
6   NKG7        0.187039   1.330464e-20
7  PTPRC        0.088257   1.665072e-16
8   LAG3        0.768926   2.863907e-16
9   CMC1        0.616737   3.124905e-15

CD4 T cells
    names  logfoldchanges     pvals_adj


In [155]:
# ==========================================================================
# DEG ANALYSIS - DYSFUNCTION
# ==========================================================================

# -- DEG between DYSFUNC-high and DYSFUNC-low per cell type
deg_results_dysfunction = {}

for cell_type in immune_obj.obs["immune_cell_type"].unique():
    subset = immune_obj[immune_obj.obs["immune_cell_type"] == cell_type].copy()

    if subset.obs["dysfunction_score"].value_counts().min() < 10:
        continue

    sc.tl.rank_genes_groups(
        subset,
        groupby="dysfunction_score",
        groups=["High"],
        reference="Low",
        method="wilcoxon",
        key_added="deg_dysfunction"
    )

    deg_results_dysfunction[cell_type] = sc.get.rank_genes_groups_df(
        subset,
        group="High",
        key="deg_dysfunction"
    )

# -- check
for cell_type, df in deg_results_dysfunction.items():
    print(f"\n{cell_type}")
    print(df[df["pvals_adj"] < 0.05].head(10)[["names", "logfoldchanges", "pvals_adj"]])

In [156]:
# ==========================================================================
# FIGURE 6_8: LABELED UMAP SHOWING CO-OCCURANCE
# ==========================================================================

centroids = pd.DataFrame(
    immune_obj.obsm["X_umap"],
    index=immune_obj.obs_names,
    columns=["UMAP1", "UMAP2"]
)
centroids["immune_cell_type"] = immune_obj.obs["immune_cell_type"].values
centroids = centroids.groupby("immune_cell_type", observed = False)[["UMAP1", "UMAP2"]].mean()



fig= sc.pl.umap(
    immune_obj,
    color="sen_dysfunction_label",
    palette=label_palette,
    title="Senescence and Dysfunction Co-occurrence",
    show=False,
    return_fig=True
)

ax = fig.axes[0]
texts = []
for cell_type, row in centroids.iterrows():
    texts.append(
        ax.text(
            row["UMAP1"],
            row["UMAP2"],
            cell_type,
            fontsize=6,
            ha="center",
            va="center",
            fontweight="bold",
        )
    )

adjust_text(texts, ax=ax)
plt.savefig(
    os.path.join(output_path_figures, f"06_umap_sen_dysfunction.png"),
    dpi=300, bbox_inches="tight"
    )
plt.close()

In [157]:
pairs = list(itertools.combinations(hue_order_tissue, 2))

pairwise_results = []

for cell_type in immune_obj.obs["immune_cell_type"].unique():
    subset = immune_obj.obs[immune_obj.obs["immune_cell_type"] == cell_type]

    for t1, t2 in pairs:
        g1 = subset[subset["tissue_type"] == t1]["dysfunction_score"].dropna()
        g2 = subset[subset["tissue_type"] == t2]["dysfunction_score"].dropna()

        if len(g1) < 10 or len(g2) < 10:
            continue

        stat, pval = mannwhitneyu(g1, g2, alternative="two-sided")

        pairwise_results.append({
            "cell_type": cell_type,
            "group1":    t1,
            "group2":    t2,
            "pval":      pval
        })

pairwise_df = pd.DataFrame(pairwise_results)

# -- FDR correction across all tests
pairwise_df["pval_adj"] = multipletests(pairwise_df["pval"], method="fdr_bh")[1]
pairwise_df["significant"] = pairwise_df["pval_adj"] < 0.05

# -- filter to significant and key comparisons
key_comparisons = pairwise_df[
    pairwise_df["group1"].isin(["Ctrl"]) |
    pairwise_df["group2"].isin(["Ctrl"])
].sort_values(["cell_type", "pval_adj"])

print(key_comparisons[["cell_type", "group1", "group2", "pval_adj", "significant"]])

                      cell_type group1 group2      pval_adj  significant
54                      B cells   Ctrl    EuE  1.719461e-02         True
56                      B cells   Ctrl    EcP  2.416932e-02         True
55                      B cells   Ctrl    EcO  1.835742e-01        False
20               CD16+ NK cells   Ctrl    EcP  2.556364e-03         True
19               CD16+ NK cells   Ctrl    EcO  2.756103e-02         True
18               CD16+ NK cells   Ctrl    EuE  9.296055e-01        False
44               CD16- NK cells   Ctrl    EcP  4.435779e-26         True
42               CD16- NK cells   Ctrl    EuE  2.979047e-07         True
43               CD16- NK cells   Ctrl    EcO  8.399246e-03         True
12                  CD4 T cells   Ctrl    EuE  1.002851e-01        False
13                  CD4 T cells   Ctrl    EcO  4.747503e-01        False
14                  CD4 T cells   Ctrl    EcP  8.456965e-01        False
6                   CD8 T cells   Ctrl    EuE  1.31

In [158]:
# ==========================================================================
# SAVING RESULTS
# ==========================================================================

# -- Overall Columns
senescence_scores = immune_obj.obs[
    [
        "sample_id",
        "patient_id",
        "tissue_type",
        "condition",
        "lesion_site",

        "immune_cell_type",
        "lineage",

        "core_sen_score",
        "sasp_score",
        "innate_sen_score",
        "adaptive_sen_score",
        "composite_sen_score",

        "dysfunction_score",

        "senescent_state",
        "dysfunction_state",
        "sen_dysfunction_label"
    ]
].copy()

senescence_scores.to_csv(os.path.join(output_path_data, "06_senescence_scores.csv"))

senescence_summary= immune_obj.obs.groupby(
["immune_cell_type", "tissue_type"],
                                  observed=False
)[["core_sen_score", "sasp_score", "composite_sen_score","dysfunction_score"]].mean().reset_index()

# -- Senescence Summary

senescence_summary = (
    senescence_summary
    .merge(
        prop_summary,
        on=["tissue_type", "immune_cell_type"],
        how="left"
    )
    .merge(
        cell_counts,
        on=["tissue_type", "immune_cell_type"],
        how="left"
    )
)


senescence_summary.to_csv(os.path.join(output_path_data, "06_senescence_summary.csv"))

# -- High Senescence Markers
deg_results_sen

df_list = []

for cell_type, metrics in deg_results_sen.items():
    temp_df = pd.DataFrame(metrics)

    temp_df.insert(0, 'cell_type', cell_type)

    df_list.append(temp_df)

senescence_high_markers = pd.concat(df_list, ignore_index=True)

senescence_high_markers.to_csv(os.path.join(output_path_data, "06_senescence_high_markers.csv"))


In [159]:
# -- Save subsetted object for neighborhood analysis
immune_obj.write_h5ad(os.path.join(output_path_data, "GSE179640_immunosenescence.h5ad"))